In [2]:
import pandas as pd
# =========================
# 0) 파일 경로만 바꿔서 실행
# =========================
TOP_RISK_PATH = "/Users/bagjimin/Downloads/integrated_top_risk.csv"          # 1번 사진에서 뽑은 초고위험군 파일
HIGH_RISK_PATH = "/Users/bagjimin/Downloads/integrated_high_risk.csv"   
CANCEL_PATH   = "/Users/bagjimin/Desktop/LG_HelloVision/PROJECT/dataset/sha_tps/sha_tps_cancel_202311_to_202312.csv"  

top = pd.read_csv(TOP_RISK_PATH)
cancel = pd.read_csv(CANCEL_PATH)
high = pd.read_csv(HIGH_RISK_PATH)

print("top columns:", top.columns.tolist())
print("cancel columns:", cancel.columns.tolist())



top columns: ['고객 ID', '취향 및 요금제 기반', '시청 관련 데이터 기반', '불만 지수 기반']
cancel columns: ['sha2_hash', 'SVC_USE_DAYS_GRP', 'MEDIA_NM_GRP', 'PROD_NM_GRP', 'PROD_OLD_YN', 'PROD_ONE_PLUS_YN', 'AGMT_KIND_NM', 'STB_RES_1M_YN', 'SVOD_SCRB_CNT_GRP', 'PAID_CHNL_CNT_GRP', 'SCRB_PATH_NM_GRP', 'INHOME_RATE', 'AGMT_END_SEG', 'AGMT_END_YMD', 'TOTAL_USED_DAYS', 'TV_SCRB', 'ANALOG_SCRB', 'DIGITAL_SCRB', 'TOTAL_INTERNET_SCRB', 'GIGA_INTERNET_SCRB', 'BUNDLE_YN', 'DIGITAL_GIGA_YN', 'DIGITAL_ALOG_YN', 'TV_I_CNT', 'CH_LAST_DAYS_BF_GRP', 'VOC_TOTAL_MONTH1_YN', 'VOC_STOP_CANCEL_MONTH1_YN', 'AGE_GRP10', 'EMAIL_RECV_CLS_NM', 'SMS_SEND_CLS_NM', 'CH_HH_AVG_MONTH1', 'CH_25_RATIO_MONTH1', 'CH_25_RATIO_MEAN_3MM', 'CH_FAV_RNK1', 'KIDS_USE_PV_MONTH1', 'NFX_USE_YN', 'YTB_USE_YN', 'p_mt', 'cancel_yn']


In [3]:
TARGET_MT = 202311
# 1️⃣ 초고위험군 고객 ID set
# ---------------------------
top_ids = set(top["고객 ID"].astype(str))
print("초고위험군 수:", len(top_ids))
high_ids = set(high["고객 ID"].astype(str))
print("고위험군 수:", len(high_ids))

초고위험군 수: 469
고위험군 수: 28483


In [4]:
# ---------------------------
# 2️⃣ 202311 데이터 필터
# ---------------------------
cancel_202311 = cancel[cancel["p_mt"] == TARGET_MT].copy()

# 1) 고객별 cancel_yn 고유값 개수 확인
status_cnt = (
    cancel_202311
    .groupby("sha2_hash")["cancel_yn"]
    .nunique()
    .reset_index(name="status_unique_cnt")
)

# 2) 유지/해지 혼재 고객 제거 (고유값이 1개인 고객만 유지)
valid_ids = set(
    status_cnt.loc[status_cnt["status_unique_cnt"] == 1, "sha2_hash"]
)

cancel_clean = cancel_202311[
    cancel_202311["sha2_hash"].isin(valid_ids)
].copy()

print("혼재 고객 제거 전:", cancel_202311["sha2_hash"].nunique())
print("혼재 고객 제거 후:", cancel_clean["sha2_hash"].nunique())

# ---------------------------
# 실제 해지자 set 생성
# ---------------------------
true_cancel = cancel_clean[cancel_clean["cancel_yn"] == "해지"]
true_cancel_ids = set(true_cancel["sha2_hash"].astype(str))

all_202311_ids = set(cancel_clean["sha2_hash"].astype(str))

print("202311 전체 고객 수 (정제 후):", len(all_202311_ids))
print("202311 실제 해지자 수 (정제 후):", len(true_cancel_ids))


혼재 고객 제거 전: 2037109
혼재 고객 제거 후: 1998550
202311 전체 고객 수 (정제 후): 1998550
202311 실제 해지자 수 (정제 후): 27931


In [5]:
# ---------------------------
# 3️⃣ 교집합
# ---------------------------
hit_ids = top_ids & true_cancel_ids

n_top = len(top_ids)
n_cancel = len(true_cancel_ids)
n_hit = len(hit_ids)

precision = n_hit / n_top if n_top else 0
recall = n_hit / n_cancel if n_cancel else 0
base_rate = n_cancel / len(all_202311_ids)
lift = precision / base_rate if base_rate else 0

print("\n===== 성능 결과 =====")
print("적중 수 (n_hit):", n_hit)
print("Precision (초고위험군 해지율):", round(precision,4))
print("Recall (해지자 중 탐지율):", round(recall,4))
print("Base rate (전체 해지율):", round(base_rate,4))
print("Lift:", round(lift,3))


===== 성능 결과 =====
적중 수 (n_hit): 93
Precision (초고위험군 해지율): 0.1983
Recall (해지자 중 탐지율): 0.0033
Base rate (전체 해지율): 0.014
Lift: 14.189


In [6]:
# ---------------------------
# 3️⃣ 고위험군 교집합
# ---------------------------
hit2_ids = high_ids & true_cancel_ids

n_top = len(high_ids)
n_cancel = len(true_cancel_ids)
n_hit = len(hit2_ids)

precision = n_hit / n_top if n_top else 0
recall = n_hit / n_cancel if n_cancel else 0
base_rate = n_cancel / len(all_202311_ids)
lift = precision / base_rate if base_rate else 0

print("\n===== 성능 결과 =====")
print("적중 수 (n_hit):", n_hit)
print("Precision (고위험군 해지율):", round(precision,4))
print("Recall (해지자 중 탐지율):", round(recall,4))
print("Base rate (전체 해지율):", round(base_rate,4))
print("Lift:", round(lift,3))


===== 성능 결과 =====
적중 수 (n_hit): 2271
Precision (고위험군 해지율): 0.0797
Recall (해지자 중 탐지율): 0.0813
Base rate (전체 해지율): 0.014
Lift: 5.705
